# CYMEK FORMATION-MUX-001 — Kaggle T4 x2

## BEFORE RUNNING

Kaggle **Settings → Accelerator → GPU T4 x2** and **Internet → ON**.

This notebook runs frozen **Science S5**. The campaign may take more than one Kaggle session: it never shortens the scientific exposure to fit a session. Near the per-session wall it stops launching new work, preserves exact-resume state, packages results, and continues from that output in the next session.

S5 uses 60,000 unique training rows (10,000 per family), 480 development rows, and 720 sealed rows. CS-MECH checkpoints every 200 updates. REP-FORM remains scientifically matched by processed tokens and checkpoints every 10,000 processed tokens. Every checkpoint writes `LATEST_PROGRESS.json` plus an immutable `progress/UPDATE_*.json` snapshot. Raw sealed examples are never persisted.

If status is `PARTIAL_SESSION`, save the Kaggle version/output, attach that output to the next run, and rerun this exact notebook.


In [ ]:
# CELL 1 — fetch and verify immutable operator + Science S5
import pathlib, subprocess, sys
REPO = pathlib.Path('/kaggle/working/An-Ra-the-new-AGI')
REMOTE = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
OPERATOR_COMMIT = '4e3bb50c683d94a09801703dfb9f469debe64a50'
OPERATOR_PATH = 'tools/formation_mux_001_kaggle_operator_v8.py'
OPERATOR_BLOB = 'b427269f682322c9ce58f1677710c5979c76465c'
SCIENCE_COMMIT_S5 = 'c15ad8beb409537db42d075684ea54847a074ebd'
if not REPO.exists():
    subprocess.run(['git', 'clone', REMOTE, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', OPERATOR_COMMIT], check=True)
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
assert head == OPERATOR_COMMIT, (head, OPERATOR_COMMIT)
blob = subprocess.run(['git', '-C', str(REPO), 'hash-object', OPERATOR_PATH], capture_output=True, text=True, check=True).stdout.strip()
assert blob == OPERATOR_BLOB, (blob, OPERATOR_BLOB)
subprocess.run(['git', '-C', str(REPO), 'cat-file', '-e', SCIENCE_COMMIT_S5 + '^{commit}'], check=True)
try:
    import tokenizers  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers'], check=True)
print('OPERATOR VERIFIED:', OPERATOR_COMMIT)
print('OPERATOR BLOB VERIFIED:', OPERATOR_BLOB)
print('SCIENCE S5 AVAILABLE:', SCIENCE_COMMIT_S5)


In [ ]:
# CELL 2 — one canonical long-run path
import pathlib, subprocess, sys
root = pathlib.Path('/kaggle/working/FORMATION_MUX_001')
code = subprocess.run([
    sys.executable, '-u', 'tools/formation_mux_001_kaggle_operator_v8.py',
    '--repo', '/kaggle/working/An-Ra-the-new-AGI',
    '--out', str(root),
], cwd='/kaggle/working/An-Ra-the-new-AGI')
if code.returncode != 0:
    failure = root / 'GLOBAL_FAILURE.json'
    if failure.exists(): print('GLOBAL_FAILURE:', failure.read_text())
    raise RuntimeError('FORMATION-MUX failed closed with exit ' + str(code.returncode) + '; preserve the Kaggle Output and result ZIP')


In [ ]:
# CELL 3 — status / checkpoint / result retrieval
import json, pathlib
root = pathlib.Path('/kaggle/working/FORMATION_MUX_001')
bundle = pathlib.Path('/kaggle/working/FORMATION_MUX_001_RESULTS.zip')
state = json.loads((root / 'CAMPAIGN_STATE.json').read_text()) if (root / 'CAMPAIGN_STATE.json').exists() else {}
print('STATUS:', state.get('status'))
print('ARMS:', state.get('complete_arms'), '/', state.get('required_arms'))
print('PENDING SEED BUNDLES:', state.get('pending_seed_bundles'))
print('WALL GUARD:', state.get('wall_guard_triggered'))
latest = sorted(root.rglob('LATEST_PROGRESS.json'))
print('LATEST PROGRESS SNAPSHOTS:', len(latest))
for p in latest:
    x = json.loads(p.read_text())
    print(p.relative_to(root), 'updates=', x.get('updates'), 'processed=', x.get('processed_tokens'), 'formation=', x.get('formation_so_far'))
for exp in ('CS-MECH-002', 'REP-FORM-003A'):
    p = root / exp / 'FINAL_RESULT.json'
    print(exp, '->', json.loads(p.read_text()).get('verdict') if p.exists() else 'not finalized')
print('RESULT ZIP:', bundle, 'exists=', bundle.exists())
if state.get('status') == 'PARTIAL_SESSION':
    print('NEXT: save this Kaggle version/output, attach that output to the next run, and rerun this exact notebook.')
